# Objective 4: Identify which stylometric features are most robust to paraphrasing attacks

Already calculated the change of features under paraphrasing so will be able to utilise that here.

In [1]:
import pandas as pd
import numpy as np

In [2]:
obj_4_changes_main = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\obj_3_changes_main.csv"
)

In [3]:
obj_4_changes_main.head(10)

,document_id,author,feature,condition,delta,abs_delta,feature_group
0,AI_AMZ_0014,AI,alpha_ratio,light,0.000470,0.000470,surface
1,AI_AMZ_0014,AI,alpha_ratio,translation,0.001851,0.001851,surface
2,AI_AMZ_0014,AI,alpha_ratio,heavy,0.009178,0.009178,surface
3,AI_AMZ_0014,AI,character_count,light,0.000000,0.000000,surface
4,AI_AMZ_0014,AI,character_count,translation,40.000000,40.000000,surface
5,AI_AMZ_0014,AI,character_count,heavy,301.000000,301.000000,surface
6,AI_AMZ_0014,AI,colon_ratio,light,0.000000,0.000000,surface
7,AI_AMZ_0014,AI,colon_ratio,translation,0.000000,0.000000,surface
8,AI_AMZ_0014,AI,colon_ratio,heavy,0.000000,0.000000,surface
9,AI_AMZ_0014,AI,comma_ratio,light,0.000940,0.000940,surface


In [4]:
print(obj_4_changes_main["condition"].value_counts())
print(obj_4_changes_main["author"].value_counts())

# group feature change data to better inspect changes
robust_summary = (
    obj_4_changes_main
    .groupby(["feature_group", "feature", "author", "condition"])
    .agg(
        median_abs_change = ("abs_delta", "median"),
        mean_abs_change = ("abs_delta", "mean"),
        std_abs_change = ("abs_delta", "std"),
        num = ("abs_delta", "count")
    )
    .reset_index()
)

robust_summary.head(10)

condition
light          159400
translation    159400
heavy          159400
Name: count, dtype: int64
author
AI       239400
human    238800
Name: count, dtype: int64


,feature_group,feature,author,condition,median_abs_change,mean_abs_change,std_abs_change,num
0,semantic,hypernym_density,AI,heavy,0.020058,0.023404,0.017542,399
1,semantic,hypernym_density,AI,light,0.007520,0.009465,0.007514,399
2,semantic,hypernym_density,AI,translation,0.008604,0.010369,0.008348,399
3,semantic,hypernym_density,human,heavy,0.021944,0.027719,0.021970,398
4,semantic,hypernym_density,human,light,0.010376,0.012760,0.010364,398
5,semantic,hypernym_density,human,translation,0.015635,0.017827,0.013359,398
6,semantic,hyponym_density,AI,heavy,0.017977,0.022380,0.016950,399
7,semantic,hyponym_density,AI,light,0.007853,0.009647,0.007959,399
8,semantic,hyponym_density,AI,translation,0.009132,0.011112,0.008965,399
9,semantic,hyponym_density,human,heavy,0.021825,0.026975,0.021002,398


Lets pivot to insepct side by side.

In [5]:
# pivot df based on median absolute change to inspect
# light > trans > heavy
robust_pivot = (
    robust_summary
    .pivot_table(
        index = ["feature_group", "feature", "author"],
        columns = "condition",
        values = "median_abs_change"
    )
    .reset_index()
)

# remove column index
robust_pivot.columns.name = None
# set order
robust_pivot = robust_pivot[
    ["feature_group", "feature", "author",
     "light", "translation", "heavy"
    ]
]

robust_pivot.head(20)

,feature_group,feature,author,light,translation,heavy
0,semantic,hypernym_density,AI,0.007520,0.008604,0.020058
1,semantic,hypernym_density,human,0.010376,0.015635,0.021944
2,semantic,hyponym_density,AI,0.007853,0.009132,0.017977
3,semantic,hyponym_density,human,0.010703,0.015630,0.021825
4,semantic,mean_adj_sent_similarity,AI,0.006118,0.008868,0.018899
5,semantic,mean_adj_sent_similarity,human,0.008578,0.011148,0.028876
6,semantic,mean_embedding_variance,AI,0.000012,0.000015,0.000045
7,semantic,mean_embedding_variance,human,0.000015,0.000024,0.000061
8,semantic,mean_sent_doc_similarity,AI,0.004541,0.005331,0.016418
9,semantic,mean_sent_doc_similarity,human,0.005391,0.008492,0.021624


We want to be able to quantify the amount of change present for each feature and this can't be done as is, as, for example, character count changes might be hundreds of words whereas another variable might be 0.00003. Percentage change not ideal due to the variance in features. Similarly, normalising via STD is also susceptible to outliers. Will therefore normalise via IQR. But to get original values to calculate this will need to re-load previous feature sheets. 

In [6]:
surface_features = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\obj_3_surface_features.csv"
)

syntactic_features = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\obj_3_syntactic_features.csv"
)

semantic_features = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\obj_3_semantic_features.csv"
)

So now we want to re-obtain the original feature values from these feature sets (should have carried over initially), sao that we can use these to calculate an interqartile range which we can use to normalise the median absolute change.

In [7]:
# metadata columns
meta_cols = [
    "document_id", "source", "topic", 
    "label", "author", "condition"
]

# only need original texts as already calculated stats for paraphrases
surface_original = surface_features[surface_features["condition"] == "original"].copy()
syntactic_original = syntactic_features[syntactic_features["condition"] == "original"].copy()
semantic_original = semantic_features[semantic_features["condition"] == "original"].copy()

# functin to convert dfs to long for easier comparison / extraction
def reshape(df, feature_group):

    # specify feature columns, i.e. those not in metadata
    feature_cols = [
        col for col in df.columns
        if col not in meta_cols
    ]
    # reshape the DF to long, also extracting original value
    reshaped_df = df.melt(
        id_vars = ["document_id", "author"],
        value_vars = feature_cols,
        var_name = "feature",
        value_name = "original_value"
    )
    # and specify feature group
    reshaped_df["feature_group"] = feature_group

    return reshaped_df


# run that function on the three feature sets...
surface_reshape = reshape(surface_original, "surface")
syntactic_reshape = reshape(syntactic_original, "syntactic")
semantic_reshape = reshape(semantic_original, "semantic")
# ...and combine to one DF
original_features_reshaped = pd.concat(
    [surface_reshape, syntactic_reshape, semantic_reshape],
    ignore_index = True
)

# get 1st and 3rd quartiles of original values...
original_iqrs = (
    original_features_reshaped
    .groupby(["feature_group", "feature", "author"])
    ["original_value"]
    .agg(
    q1 = lambda x : x.quantile(0.25),
    q3 = lambda x : x.quantile(0.75)
    )
    .reset_index()
)
# ... and use to calculate IQR
original_iqrs["original_iqr"] = (
    original_iqrs["q3"] - original_iqrs["q1"]
)

# now merge these IQR values into the original robust DF
robust_norm = robust_summary.merge(
    original_iqrs[
        ["feature_group", "feature", "author", "original_iqr"]
    ],
    on = ["feature_group", "feature", "author"],
    # ensuring to keep all original rows from robust summary
    how = "left",
    validate = "many_to_one"
)

# now can calulate the normalised IQR change using the 
# median absolute change...
robust_norm["iqr_norm_change"] = (
    robust_norm["median_abs_change"]
    / robust_norm["original_iqr"]
)
# ... but set to nan if original iqr = 0
robust_norm.loc[
    robust_norm["original_iqr"] == 0,
    "iqr_norm_change"
] = np.nan

# sort
robust_norm = robust_norm.sort_values(
    ["condition", "author", "iqr_norm_change"],
    ascending = [True, True, True],
    na_position = "last"
).reset_index(drop = True)

# print(robust_norm.shape)
robust_norm.head(20)   

,feature_group,feature,author,condition,median_abs_change,mean_abs_change,std_abs_change,num,original_iqr,iqr_norm_change
0,surface,func_ratio_do,AI,heavy,0.000000,0.001007,0.001664,399,0.002421,0.000000
1,surface,func_ratio_every,AI,heavy,0.000000,0.000921,0.001352,399,0.002415,0.000000
2,surface,func_ratio_my,AI,heavy,0.000000,0.000860,0.001574,399,0.004872,0.000000
3,surface,func_ratio_up,AI,heavy,0.000000,0.001138,0.001622,399,0.002404,0.000000
4,surface,func_ratio_what,AI,heavy,0.000000,0.001033,0.001531,399,0.002439,0.000000
5,surface,func_ratio_will,AI,heavy,0.000000,0.000963,0.001655,399,0.002618,0.000000
6,surface,question_ratio,AI,heavy,0.000000,0.000071,0.000184,399,0.000405,0.000000
7,surface,colon_ratio,AI,heavy,0.000005,0.000165,0.000230,399,0.000368,0.013995
8,syntactic,dependency_ratio_flat,AI,heavy,0.000087,0.000513,0.000987,399,0.004162,0.020805
9,syntactic,pos_ratio_PROPN,AI,heavy,0.001296,0.002157,0.002851,399,0.029771,0.043533


In [8]:
print(robust_norm["iqr_norm_change"].isna().sum())

138


Before looking into the normalised median absolute changes which should point toward features that do not change much under paraphrasing, implying robustness, we should first look for these NaNs.

In [9]:
nan_norm = robust_norm[robust_norm["iqr_norm_change"].isna()]
nan_norm.head(20)

,feature_group,feature,author,condition,median_abs_change,mean_abs_change,std_abs_change,num,original_iqr,iqr_norm_change
171,surface,emdash_ratio,AI,heavy,0.0,0.000217,0.000335,399,0.0,NaN
172,surface,endash_ratio,AI,heavy,0.0,0.000010,0.000064,399,0.0,NaN
173,surface,exclamation_ratio,AI,heavy,0.0,0.000001,0.000023,399,0.0,NaN
174,surface,func_ratio_all,AI,heavy,0.0,0.000949,0.001322,399,0.0,NaN
175,surface,func_ratio_any,AI,heavy,0.0,0.000524,0.000985,399,0.0,NaN
176,surface,func_ratio_down,AI,heavy,0.0,0.000607,0.001176,399,0.0,NaN
177,surface,func_ratio_even,AI,heavy,0.0,0.000879,0.001339,399,0.0,NaN
178,surface,func_ratio_her,AI,heavy,0.0,0.000723,0.001676,399,0.0,NaN
179,surface,func_ratio_his,AI,heavy,0.0,0.000648,0.001483,399,0.0,NaN
180,surface,func_ratio_must,AI,heavy,0.0,0.000614,0.001180,399,0.0,NaN


It seems that in these cases the iqr normalised change is NaN because the original IQR is 0. Just double check how many this accounts for from the 138 total.

In [10]:
print(
    (nan_norm["original_iqr"] == 0)
    .sum()
)

138


So all NaNs are due to a zero IQR. This does not mean they are robust, or otherwise, but we should create a new DF to exclude these from the analysis of IQR normalised change.

In [11]:
robust_norm_valid = robust_norm[
    robust_norm["iqr_norm_change"]
    .notna()
]
print(robust_norm.shape)
print(robust_norm_valid.shape)

(1200, 10)
(1062, 10)


Now want to pivot it back again (!) so that we can inspect light > trans > heavy side by side for change normalised by IQR.

In [12]:
# pivot the table
rnv_pivot = robust_norm_valid.pivot_table(
    index = ["feature_group", "feature", "author"],
    # making condition values column heads
    columns = "condition",
    # where IQR norm change is the value we are interested in
    values = "iqr_norm_change"
).reset_index()

# drop old column which became index
rnv_pivot.columns.name = None

# specify column order
rnv_pivot = rnv_pivot[
    [
        "feature_group", "feature", "author",
         "light", "translation", "heavy"
    ]
]

rnv_pivot.head(20)

,feature_group,feature,author,light,translation,heavy
0,semantic,hypernym_density,AI,0.058420,0.066845,0.155828
1,semantic,hypernym_density,human,0.132827,0.200152,0.280908
2,semantic,hyponym_density,AI,0.069197,0.080468,0.158410
3,semantic,hyponym_density,human,0.122317,0.178628,0.249425
4,semantic,mean_adj_sent_similarity,AI,0.063386,0.091878,0.195800
5,semantic,mean_adj_sent_similarity,human,0.077701,0.100984,0.261565
6,semantic,mean_embedding_variance,AI,0.053104,0.065605,0.193496
7,semantic,mean_embedding_variance,human,0.058477,0.091624,0.235984
8,semantic,mean_sent_doc_similarity,AI,0.056101,0.065871,0.202842
9,semantic,mean_sent_doc_similarity,human,0.059531,0.093765,0.238775


We can now see the IQR normalised change for each feature and author combination across light > trans > heavy, but now we want to take a mean across these values to see which were overall most robust to paraphrasing (by showing the least change).

In [13]:
rnv_pivot["mean_norm_change"] = (
    rnv_pivot[["light", "translation", "heavy"]]
    .mean(axis = 1)
)

rnv_pivot = rnv_pivot.sort_values(
    "mean_norm_change",
    ascending = True
).reset_index(drop = True)

print(rnv_pivot.shape)
rnv_pivot.head(30)

(354, 7)


,feature_group,feature,author,light,translation,heavy,mean_norm_change
0,surface,func_ratio_will,AI,0.000000,0.000000,0.000000,0.000000
1,surface,func_ratio_do,AI,0.000000,0.000000,0.000000,0.000000
2,surface,func_ratio_our,human,0.000000,0.000000,0.000000,0.000000
3,surface,func_ratio_down,human,0.000000,0.000000,0.000000,0.000000
4,surface,func_ratio_will,human,0.000000,0.000000,0.000000,0.000000
5,syntactic,pos_ratio_SYM,human,0.000000,0.000000,0.000000,0.000000
6,surface,func_ratio_up,AI,0.000000,0.000000,0.000000,0.000000
7,surface,func_ratio_even,human,0.000000,0.000000,0.000000,0.000000
8,surface,func_ratio_your,human,0.000000,0.000000,0.000000,0.000000
9,surface,exclamation_ratio,human,0.000000,0.000000,0.000000,0.000000


In [16]:
rnv_pivot.head(50)

,feature_group,feature,author,light,translation,heavy,mean_norm_change
0,surface,func_ratio_will,AI,0.000000,0.000000,0.000000,0.000000
1,surface,func_ratio_do,AI,0.000000,0.000000,0.000000,0.000000
2,surface,func_ratio_our,human,0.000000,0.000000,0.000000,0.000000
3,surface,func_ratio_down,human,0.000000,0.000000,0.000000,0.000000
4,surface,func_ratio_will,human,0.000000,0.000000,0.000000,0.000000
5,syntactic,pos_ratio_SYM,human,0.000000,0.000000,0.000000,0.000000
6,surface,func_ratio_up,AI,0.000000,0.000000,0.000000,0.000000
7,surface,func_ratio_even,human,0.000000,0.000000,0.000000,0.000000
8,surface,func_ratio_your,human,0.000000,0.000000,0.000000,0.000000
9,surface,exclamation_ratio,human,0.000000,0.000000,0.000000,0.000000


This is useful, after row 21. Before that we observe a mean normalised change of zero due to zero values for light, trans and heavy, which is due to zero values of iqr normalised change for that feature. So we should inspect where these values came from to determine whether they are based on a median of zero due to a specific spread of values, or a median of zero due to no / low usage.

In [17]:
# but first lets save this as this is the ranked list of the 'most robust' features

robust_combined.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\objective_4_most_robust_features.csv", 
    index = False
)

____

In [47]:
# make new df for these features where mean norm change = 0
zero_features = rnv_pivot[rnv_pivot["mean_norm_change"] == 0]

# and extract these features from the objective 4 main changes df
# to observe the changes at an individual document level
zero_changes = obj_4_changes_main.merge(
    zero_features[["feature_group", "feature", "author"]],
    on = ["feature_group", "feature", "author"],
    how = "inner",
    validate = "many_to_one"
)

# and now make a summary of these changes at the document level
zero_change_sum = (
    zero_changes
    .groupby(["feature_group", "feature", "author", "condition"])
    .agg(
        num_compared = ("abs_delta", "count"),
        num_changed = ("abs_delta", lambda x : (x > 0).sum()),
        pc_changed = ("abs_delta", lambda x : (x > 0).mean() * 100),
        median_abs_change = ("abs_delta", "median"),
        mean_abs_change = ("abs_delta", "mean"),
        max_abs_change = ("abs_delta", "max")
    )
    .reset_index()
)

# finally order the summary by those whose % changed the most
# followed by maximum absolute change
zero_change_sum = zero_change_sum.sort_values(
    ["pc_changed", "max_abs_change"],
    ascending = [False, False]
).reset_index(drop = True)    

zero_change_sum

,feature_group,feature,author,condition,num_compared,num_changed,pc_changed,median_abs_change,mean_abs_change,max_abs_change
0,surface,func_ratio_will,human,heavy,398,192,48.241206,0.0,0.001582,0.013850
1,surface,func_ratio_will,human,translation,398,191,47.989950,0.0,0.001072,0.011461
2,surface,func_ratio_every,AI,heavy,399,187,46.867168,0.0,0.000921,0.006424
3,surface,func_ratio_his,human,heavy,398,184,46.231156,0.0,0.001592,0.015031
4,surface,func_ratio_what,AI,heavy,399,184,46.115288,0.0,0.001033,0.007828
...,...,...,...,...,...,...,...,...,...,...
61,surface,exclamation_ratio,human,translation,398,107,26.884422,0.0,0.000031,0.000868
62,surface,exclamation_ratio,human,heavy,398,106,26.633166,0.0,0.000248,0.008954
63,surface,func_ratio_down,human,light,398,105,26.381910,0.0,0.000425,0.010444
64,surface,exclamation_ratio,human,light,398,104,26.130653,0.0,0.000010,0.000478


Now we can see that, for example, despite the feature func_ratio_will showing a mean normalised change of zero, it actually experienced change in 48% of the pieces of texts analuysed. And even the lowest ranked feature, func_ratio_our, again with a mean absolute change of zero, still changed in 25% of individual texts.

_____

We've observed that even when there is zero mean change, that is due to > 50% of documents not showing a change in feature, yet there are still many dcuments experiencing change. Will return to this later, for now, returning to the main robust feature analysis. And we will retain these, as defining robustness is not the same as defining usefullness.

In [52]:
# lets group into human vs AI top features
human_robust = (
    rnv_pivot[rnv_pivot["author"] == "human"]
    .reset_index(drop = True)
)

ai_robust = (
    rnv_pivot[rnv_pivot["author"] == "AI"]
    .reset_index(drop = True)
)

display(human_robust.head(20))
display(ai_robust.head(20))

,feature_group,feature,author,light,translation,heavy,mean_norm_change
0,surface,question_ratio,human,0.000000,0.000000,0.000000,0.000000
1,syntactic,dependency_ratio_discourse,human,0.000000,0.000000,0.000000,0.000000
2,surface,func_ratio_now,human,0.000000,0.000000,0.000000,0.000000
3,surface,func_ratio_may,human,0.000000,0.000000,0.000000,0.000000
4,surface,func_ratio_any,human,0.000000,0.000000,0.000000,0.000000
5,surface,func_ratio_his,human,0.000000,0.000000,0.000000,0.000000
6,syntactic,pos_ratio_INTJ,human,0.000000,0.000000,0.000000,0.000000
7,surface,exclamation_ratio,human,0.000000,0.000000,0.000000,0.000000
8,surface,func_ratio_your,human,0.000000,0.000000,0.000000,0.000000
9,surface,func_ratio_even,human,0.000000,0.000000,0.000000,0.000000


,feature_group,feature,author,light,translation,heavy,mean_norm_change
0,surface,func_ratio_will,AI,0.000000,0.000000,0.000000,0.000000
1,surface,func_ratio_my,AI,0.000000,0.000000,0.000000,0.000000
2,surface,func_ratio_every,AI,0.000000,0.000000,0.000000,0.000000
3,surface,func_ratio_what,AI,0.000000,0.000000,0.000000,0.000000
4,surface,question_ratio,AI,0.000000,0.000000,0.000000,0.000000
5,surface,func_ratio_up,AI,0.000000,0.000000,0.000000,0.000000
6,surface,func_ratio_do,AI,0.000000,0.000000,0.000000,0.000000
7,surface,colon_ratio,AI,0.000000,0.000000,0.013995,0.004665
8,syntactic,dependency_ratio_flat,AI,0.000000,0.000000,0.020805,0.006935
9,surface,func_ratio_who,AI,0.000000,0.000000,0.054632,0.018211


Now we can look at robustness by paraphrase condition and feature group to look for patterns.

In [57]:
# create new df
robust_by_group = (
    rnv_pivot
    # grouping by feature group and author
    .groupby(["feature_group", "author"])
    .agg(
        # and taking means and medians across all conditions and authors
        mean_light = ("light", "mean"),
        mean_translation = ("translation", "mean"),
        mean_heavy = ("heavy", "mean"),
        median_light = ("light", "median"),
        median_translation = ("translation", "median"),
        median_heavy = ("heavy", "median"),
        # and number of unique features
        num_features = ("feature", "nunique")
    )
    .reset_index()
)

# create order to enable sort
order = ["surface", "syntactic", "semantic"]
# and set feature group to be categorical
robust_by_group["feature_group"] = pd.Categorical(
    robust_by_group["feature_group"],
    categories = order,
    ordered = True,
)
# then can sort by feature group and author
robust_by_group = robust_by_group.sort_values(
    ["author", "feature_group"]
).reset_index(drop = True)

robust_by_group

,feature_group,author,mean_light,mean_translation,mean_heavy,median_light,median_translation,median_heavy,num_features
0,surface,AI,0.061673,0.095607,0.344419,0.007764,0.042451,0.309767,77
1,syntactic,AI,0.100569,0.139625,0.290718,0.087896,0.142077,0.277748,87
2,semantic,AI,0.086503,0.095289,0.230387,0.063386,0.080468,0.195800,7
3,surface,human,0.084736,0.145902,0.362091,0.018497,0.070952,0.382921,86
4,syntactic,human,0.163731,0.208948,0.349943,0.161525,0.214648,0.354322,90
5,semantic,human,0.108508,0.155338,0.285768,0.122317,0.178628,0.261565,7


Unique features reduced to varying degrees due to earlier pipeline process removing NaNs etc. Clearly see that for all values the normalised change increases in all cases from light > trans > heavy, for both means and medians across all feature groups and authors , and additionally, that for equivalent readings, the normalised change is always higher for the human rather than AI authored text, implying some underlying reason.

Numbers above are medians and means for all features within a group. Now lets look at the best performing features under different feature and author groups.

In [60]:
# loop through human and AI author
for author_name in ["AI", "human"]:
    # and get data for that author from main df
    author_data = rnv_pivot[
    rnv_pivot["author"] == author_name
    ]
    
    print(f"\nBest performing features for {author_name} texts.")

    # loop through conditions
    for condition in ["light", "translation", "heavy"]:
        
        print(f"\n{condition} paraphrasing:")

        # and for each condition display stats
        display(
            author_data
            .sort_values(condition, ascending = True)
            [
                ["feature_group", "feature", condition]
            ]
            .head(10)
        )


Best performing features for AI texts.

light paraphrasing:


,feature_group,feature,light
0,surface,func_ratio_will,0.0
74,syntactic,sent_with_clausal_dep_ratio,0.0
63,surface,func_ratio_were,0.0
211,surface,func_ratio_their,0.0
61,syntactic,dependency_ratio_csubj,0.0
60,surface,func_ratio_so,0.0
52,syntactic,dependency_ratio_expl,0.0
190,surface,sentence_count,0.0
49,surface,func_ratio_there,0.0
48,surface,func_ratio_then,0.0



translation paraphrasing:


,feature_group,feature,translation
0,surface,func_ratio_will,0.0
61,syntactic,dependency_ratio_csubj,0.0
49,surface,func_ratio_there,0.0
48,surface,func_ratio_then,0.0
43,surface,func_ratio_no,0.0
39,surface,func_ratio_only,0.0
190,surface,sentence_count,0.0
89,surface,func_ratio_may,0.0
60,surface,func_ratio_so,0.0
29,surface,func_ratio_who,0.0



heavy paraphrasing:


...

,feature_group,feature,light
1,surface,question_ratio,0.0
210,surface,colon_ratio,0.0
220,surface,sentence_count,0.0
226,surface,func_ratio_its,0.0
288,surface,func_ratio_also,0.0
41,surface,func_ratio_who,0.0
38,syntactic,dependency_ratio_csubj,0.0
36,surface,func_ratio_into,0.0
34,surface,func_ratio_up,0.0
33,surface,func_ratio_what,0.0



translation paraphrasing:


,feature_group,feature,translation
1,surface,question_ratio,0.0
226,surface,func_ratio_its,0.0
26,surface,func_ratio_some,0.0
288,surface,func_ratio_also,0.0
24,surface,func_ratio_only,0.0
21,surface,func_ratio_my,0.0
19,surface,func_ratio_our,0.0
18,surface,func_ratio_down,0.0
17,surface,func_ratio_will,0.0
16,syntactic,pos_ratio_SYM,0.0



heavy paraphrasing:


,feature_group,feature,heavy
1,surface,question_ratio,0.0
21,surface,func_ratio_my,0.0
19,surface,func_ratio_our,0.0
18,surface,func_ratio_down,0.0
17,surface,func_ratio_will,0.0
16,syntactic,pos_ratio_SYM,0.0
14,surface,func_ratio_even,0.0
13,surface,func_ratio_your,0.0
8,syntactic,pos_ratio_INTJ,0.0
7,surface,func_ratio_his,0.0


This is good, and useful to observe the absolute 'best performing features' however as we have already observed, these zeroes are due to < 50% usage across texts. So whilst still accurate lets also look at this list again without the zeroes.

In [61]:
# loop through human and AI author
for author_name in ["AI", "human"]:
    # and get data for that author from main df
    author_data = rnv_pivot[
    rnv_pivot["author"] == author_name
    ]
    
    print(f"\nBest performing features for {author_name} texts.")

    # loop through conditions
    for condition in ["light", "translation", "heavy"]:

        # add this in to drop zeros
        not_zero_data = author_data[author_data[condition] > 0]
        
        print(f"\n{condition} paraphrasing:")

        # and for each condition display stats
        display(
            not_zero_data
            .sort_values(condition, ascending = True)
            [
                ["feature_group", "feature", condition]
            ]
            .head(10)
        )


Best performing features for AI texts.

light paraphrasing:


,feature_group,feature,light
32,surface,func_ratio_would,0.001054
53,surface,func_ratio_had,0.001160
69,syntactic,dependency_ratio_appos,0.001327
59,surface,func_ratio_if,0.001640
35,surface,digit_ratio,0.001736
58,surface,func_ratio_some,0.002389
133,surface,func_ratio_its,0.002525
31,syntactic,pos_ratio_PROPN,0.002853
130,surface,func_ratio_one,0.003073
115,syntactic,dependency_ratio_parataxis,0.003598



translation paraphrasing:


,feature_group,feature,translation
35,surface,digit_ratio,0.006899
59,surface,func_ratio_if,0.007534
63,surface,func_ratio_were,0.009946
32,surface,func_ratio_would,0.010400
52,syntactic,dependency_ratio_expl,0.011109
53,surface,func_ratio_had,0.011640
74,syntactic,sent_with_clausal_dep_ratio,0.012857
50,surface,func_ratio_or,0.013769
31,syntactic,pos_ratio_PROPN,0.016027
77,surface,func_ratio_was,0.017100



heavy paraphrasing:


...

,feature_group,feature,light
261,surface,func_ratio_been,0.001309
42,surface,func_ratio_can,0.001314
56,surface,func_ratio_their,0.001803
47,surface,func_ratio_no,0.002382
27,surface,func_ratio_do,0.003525
37,surface,digit_ratio,0.004682
40,surface,func_ratio_than,0.005225
28,syntactic,dependency_ratio_flat,0.005654
54,surface,func_ratio_would,0.005771
30,surface,func_ratio_if,0.006097



translation paraphrasing:


,feature_group,feature,translation
38,syntactic,dependency_ratio_csubj,0.001797
27,surface,func_ratio_do,0.002594
25,surface,func_ratio_then,0.004819
42,surface,func_ratio_can,0.005139
33,surface,func_ratio_what,0.007430
41,surface,func_ratio_who,0.007897
30,surface,func_ratio_if,0.013799
28,syntactic,dependency_ratio_flat,0.014913
36,surface,func_ratio_into,0.016162
45,surface,semicolon_ratio,0.016565



heavy paraphrasing:


,feature_group,feature,heavy
28,syntactic,dependency_ratio_flat,0.024713
30,surface,func_ratio_if,0.035394
27,surface,func_ratio_do,0.038876
25,surface,func_ratio_then,0.039049
24,surface,func_ratio_only,0.039712
26,surface,func_ratio_some,0.044503
46,syntactic,pos_ratio_PROPN,0.053709
33,surface,func_ratio_what,0.060575
40,surface,func_ratio_than,0.066394
34,surface,func_ratio_up,0.070379
